# Tutorial 03 — Advanced Features

This notebook covers everything beyond the basics — all the power features of lllm:

| Section | Concepts covered |
|---------|------------------|
| 0 | Environment setup |
| 1 | Tools deep dive: `@tool`, `Function`, processors |
| 2 | Custom error handling in tools |
| 3 | `Function` with manual JSON schema |
| 4 | MCP (Model Context Protocol) servers |
| 5 | Proxy system: defining proxies and wiring via config |
| 5a | Interpreter mode: the `run_python` / `CALL_API` loop in detail |
| 5b | Jupyter Notebook Sandbox: `JupyterSession`, cell tags, error recovery |
| 6 | `lllm.toml`, YAML config, auto-discovery |
| 7 | Config inheritance and `vendor_config` |
| 8 | Skills: progressive capability disclosure |
| 9 | Creating local skills |
| 10 | Named runtimes for parallel experiments |
| 11 | Streaming responses |
| 12 | Image (multimodal) input |
| 13 | Custom exception and interrupt handlers |
| 14 | High-throughput batch processing |
| 15 | Packaged tactic from `packages/` |
| 16 | Architecture diagram |

---
## Section 0 — Environment Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env", override=False)

has_openai = bool(os.getenv("OPENAI_API_KEY"))
has_anthropic = bool(os.getenv("ANTHROPIC_API_KEY"))

if not (has_openai or has_anthropic):
    raise EnvironmentError(
        "No API key found. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your .env file."
    )

DEFAULT_MODEL = "gpt-4o-mini" if has_openai else "claude-haiku-4-5-20251001"
SMART_MODEL   = "gpt-4o"      if has_openai else "claude-sonnet-4-6"
print(f"Default model: {DEFAULT_MODEL}")
print(f"Smart model  : {SMART_MODEL}")

In [ ]:
import sys, math, json, tempfile, os, textwrap
sys.path.insert(0, "..")

from lllm import Agent, Prompt, Tactic
from lllm.invokers import build_invoker
from helpers.mock_data import SAMPLE_PRODUCTS, SAMPLE_REVIEWS, SAMPLE_TASKS
from helpers.display import print_section, print_response

---
## Section 1 — Tools Deep Dive

The `@tool` decorator turns any Python function into an LLM-callable tool.  
It inspects the function signature to build the JSON schema that gets sent to the model.

Type hint → JSON schema mapping:

| Python | JSON Schema |
|--------|-------------|
| `str` | `"string"` |
| `int` | `"integer"` |
| `float` | `"number"` |
| `bool` | `"boolean"` |
| `list` | `"array"` |
| `dict` | `"object"` |

Parameters **without** defaults are marked as `required` automatically.

In [ ]:
from lllm.core.prompt import tool, Function

# --- 1a. Basic @tool with prop_desc ---
@tool(
    description="Evaluate a safe mathematical expression and return the result as a float.",
    prop_desc={
        "expression": "A valid Python math expression using standard operators and math.* functions",
    },
)
def calculate(expression: str) -> float:
    safe_globals = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    return float(eval(expression, {"__builtins__": {}}, safe_globals))  # noqa: S307


@tool(
    description="Look up a product by its ID and return its full details.",
    prop_desc={
        "product_id": "Product ID string, e.g. 'p001'",
    },
)
def lookup_product(product_id: str) -> dict:
    for p in SAMPLE_PRODUCTS:
        if p["id"] == product_id:
            return p
    return {"error": f"Product '{product_id}' not found"}


@tool(
    description="Search products by category and optional minimum rating.",
    prop_desc={
        "category": "Product category, e.g. 'Electronics'",
        "min_rating": "Minimum rating (0.0 to 5.0), default is 0.0",
    },
)
def search_products(category: str, min_rating: float = 0.0) -> list:
    return [
        {"id": p["id"], "name": p["name"], "price": p["price"], "rating": p["rating"]}
        for p in SAMPLE_PRODUCTS
        if p["category"] == category and p["rating"] >= min_rating
    ]


print("Tools defined: calculate, lookup_product, search_products")

In [ ]:
# Attach tools to a Prompt via function_list
shopping_prompt = Prompt(
    path="tutorial/shopping_assistant",
    prompt=(
        "You are a helpful shopping assistant. Use your tools to answer product questions. "
        "Always show prices in USD."
    ),
    function_list=[lookup_product, search_products, calculate],
)

shopping_agent = Agent(
    name="shopper",
    system_prompt=shopping_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    max_interrupt_steps=5,  # allow up to 5 tool calls per respond() cycle
)

shopping_agent.open("chat")
shopping_agent.receive(
    "What Electronics products do you have rated 4.5 or above? "
    "Also, what would the total cost be if I bought all of them?"
)
r = shopping_agent.respond()
print(r.content)

In [ ]:
# Inspect whether the message involved tool calls
print("Was a function call:", r.is_function_call)
# The agent's dialog contains the full tool-call trace
dialog = shopping_agent.current_dialog
print(f"Dialog has {len(dialog.messages)} messages (including tool call/result pairs)")

In [ ]:
# --- 1b. max_interrupt_steps cap ---
# When the limit is reached, the agent is told to stop calling tools and write a final answer.
# max_interrupt_steps=0 means unlimited — avoid in production.

capped_agent = Agent(
    name="capped_shopper",
    system_prompt=shopping_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    max_interrupt_steps=2,  # only allow 2 tool calls, then force final answer
)

capped_agent.open("chat")
capped_agent.receive("What is the cheapest Electronics product? What about the most expensive?")
r = capped_agent.respond()
print(r.content)

---
## Section 2 — Tool Error Handling and Custom Processors

If a tool raises an exception, lllm catches it and feeds the error message back to the model as the tool result.  
The model can then decide how to proceed (e.g., try different arguments).

A **processor** lets you control exactly how the tool result is formatted before being shown to the model.

In [ ]:
# Tool that raises on bad input — lllm feeds the error to the model
@tool(description="Divide two numbers.", prop_desc={"a": "Numerator", "b": "Denominator"})
def safe_divide(a: float, b: float) -> str:
    if b == 0:
        raise ValueError("Division by zero is not defined.")
    return str(round(a / b, 6))


math_agent = Agent(
    name="math_bot",
    system_prompt=Prompt(
        path="tutorial/math_bot",
        prompt="You are a math assistant. Use your tools.",
        function_list=[safe_divide],
    ),
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    max_interrupt_steps=3,
    max_exception_retry=2,  # retry up to 2 times on ParseError
)

math_agent.open("calc")
math_agent.receive("What is 10 divided by 0? Explain what happens.")
r = math_agent.respond()
print(r.content)

In [ ]:
# Custom result processor — control how the tool output is shown to the model
def compact_processor(result, function_call) -> str:
    """Show only a compact one-liner instead of the verbose default format."""
    return f"[{function_call.name}] → {json.dumps(result, default=str)[:200]}"


@tool(
    description="Look up product details by ID.",
    prop_desc={"product_id": "Product ID"},
    processor=compact_processor,
)
def lookup_product_compact(product_id: str) -> dict:
    for p in SAMPLE_PRODUCTS:
        if p["id"] == product_id:
            return p
    return {"error": f"Not found: {product_id}"}


compact_agent = Agent(
    name="compact",
    system_prompt=Prompt(
        path="tutorial/compact_agent",
        prompt="You are a product assistant. Use lookup_product_compact.",
        function_list=[lookup_product_compact],
    ),
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    max_interrupt_steps=3,
)

compact_agent.open("chat")
compact_agent.receive("Tell me about product p001 and p003.")
r = compact_agent.respond()
print(r.content)

---
## Section 3 — `Function` with Manual JSON Schema

For complex schemas (nested objects, enums, `oneOf`) or when you want to declare the schema separately from the implementation, use `Function` directly.

In [ ]:
from lllm.core.prompt import Function

# Define the schema manually
order_fn = Function(
    name="create_order",
    description="Create a product order for a customer.",
    properties={
        "product_id": {"type": "string", "description": "Product ID (e.g. p001)"},
        "quantity": {"type": "integer", "description": "Number of units, minimum 1"},
        "shipping": {
            "type": "string",
            "enum": ["standard", "express", "overnight"],
            "description": "Shipping speed",
        },
        "gift_wrap": {"type": "boolean", "description": "Whether to gift-wrap the order"},
    },
    required=["product_id", "quantity"],  # shipping and gift_wrap are optional
)

# Link the implementation separately
def _create_order_impl(product_id: str, quantity: int,
                       shipping: str = "standard", gift_wrap: bool = False) -> dict:
    product = next((p for p in SAMPLE_PRODUCTS if p["id"] == product_id), None)
    if not product:
        return {"error": f"Product {product_id} not found"}
    total = product["price"] * quantity
    shipping_fees = {"standard": 0, "express": 9.99, "overnight": 24.99}
    return {
        "order_id": f"ORD-{product_id}-{quantity:04d}",
        "product": product["name"],
        "quantity": quantity,
        "subtotal": total,
        "shipping_fee": shipping_fees.get(shipping, 0),
        "total": total + shipping_fees.get(shipping, 0),
        "gift_wrap": gift_wrap,
    }

order_fn.link_function(_create_order_impl)

# Use it in a prompt
order_prompt = Prompt(
    path="tutorial/order_assistant",
    prompt="You are a shopping assistant. Help customers place orders.",
    function_list=[order_fn],
)

order_agent = Agent(
    name="order_bot",
    system_prompt=order_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    max_interrupt_steps=4,
)

order_agent.open("order")
order_agent.receive(
    "I'd like to order 2 units of product p004 with express shipping and gift wrapping please."
)
r = order_agent.respond()
print(r.content)

---
## Section 4 — MCP (Model Context Protocol) Servers

For tools exposed via the Model Context Protocol, use the `MCP` class.  
The agent discovers available tools dynamically from the MCP server.

> **Note:** MCP support requires an MCP-compatible invoker. The code below shows the configuration pattern — you need a running MCP server to execute it.

In [ ]:
from lllm.core.prompt import MCP

# Configure an MCP server connection
mcp_server = MCP(
    server_label="my_tools",
    server_url="http://localhost:8080",     # URL of your MCP server
    require_approval="never",              # 'never' | 'always' | 'untrusted'
    allowed_tools=["search", "fetch_url"],  # restrict which tools are exposed
)

mcp_prompt = Prompt(
    path="tutorial/mcp_agent",
    prompt="You are a research assistant with web access via MCP.",
    mcp_servers_list=[mcp_server],
)

# Agent construction is the same — MCP tools appear alongside regular tools
# mcp_agent = Agent(
#     name="mcp_agent",
#     system_prompt=mcp_prompt,
#     model=SMART_MODEL,
#     llm_invoker=build_invoker({"invoker": "litellm"}),
#     max_interrupt_steps=10,
# )

print("MCP server configured:")
print(f"  Label          : {mcp_server.server_label}")
print(f"  URL            : {mcp_server.server_url}")
print(f"  Allowed tools  : {mcp_server.allowed_tools}")
print("\n(Start an MCP server and uncomment the agent creation to run this.)

---
## Section 5 — Proxy System: Lazy API Discovery

The proxy system solves the scaling problem with tools: if you have many API endpoints, defining a `@tool` for each one bloats the context window and forces the model to know everything upfront.

**The two-tool pattern:**
- `query_api_doc(proxy_name)` — retrieve endpoint documentation on demand (lazy)
- `run_python(code)` — execute Python with `CALL_API(path, params)` pre-injected

The agent learns what endpoints exist just-in-time, then calls them through `CALL_API`.

In [ ]:
from lllm.proxies import BaseProxy, ProxyRegistrator

# Define a proxy wrapping a mock product catalog API
@ProxyRegistrator(
    path="catalog",
    name="Product Catalog API",
    description="Access product information, reviews, and pricing.",
)
class CatalogProxy(BaseProxy):

    @BaseProxy.endpoint(
        category="products",
        endpoint="list",
        description="List all products, optionally filtered by category.",
        params={
            "category": (str, "Electronics"),   # optional — no * suffix
            "min_rating": (float, 4.0),
        },
        response=[{"id": "p001", "name": "...", "price": 299.99}],
    )
    def list_products(self, params: dict) -> list:
        category = params.get("category")
        min_rating = float(params.get("min_rating", 0.0))
        return [
            {"id": p["id"], "name": p["name"], "price": p["price"], "rating": p["rating"]}
            for p in SAMPLE_PRODUCTS
            if (not category or p["category"] == category) and p["rating"] >= min_rating
        ]

    @BaseProxy.endpoint(
        category="products",
        endpoint="detail",
        description="Get full details for a single product.",
        params={
            "product_id*": (str, "p001"),   # * = required
        },
        response={"id": "p001", "name": "...", "price": 299.99, "description": "..."},
    )
    def get_product(self, params: dict) -> dict:
        pid = params["product_id"]
        for p in SAMPLE_PRODUCTS:
            if p["id"] == pid:
                return p
        return {"error": f"Product {pid} not found"}

    @BaseProxy.endpoint(
        category="reviews",
        endpoint="list",
        description="Get reviews for a specific product.",
        params={"product_id*": (str, "p001")},
        response=[{"reviewer": "alice", "rating": 5, "text": "..."}],
    )
    def get_reviews(self, params: dict) -> list:
        pid = params["product_id"]
        return [r for r in SAMPLE_REVIEWS if r["product_id"] == pid]


print("CatalogProxy defined with endpoints: list_products, get_product, get_reviews")

In [ ]:
# Wire the proxy to an agent via config
# When exec_env=interpreter, lllm injects query_api_doc and run_python automatically

class CatalogTactic(Tactic):
    name = "catalog_tactic"
    agent_group = ["catalog_analyst"]

    def call(self, task: str, **kwargs) -> str:
        agent = self.agents["catalog_analyst"]
        agent.open("main")
        agent.receive(task)
        return agent.respond().content


catalog_config = {
    "global": {
        "model_name": DEFAULT_MODEL,
        "model_args": {"temperature": 0.1},
    },
    "agent_configs": [
        {
            "name": "catalog_analyst",
            "system_prompt": (
                "You are a product catalog analyst. "
                "Use the catalog API tools to answer questions. "
                "Always call query_api_doc first to understand what's available."
            ),
            "proxy": {
                "activate_proxies": ["catalog"],
                "exec_env": "interpreter",     # injects query_api_doc + run_python
                "max_output_chars": 5000,
                "timeout": 30.0,
            },
        },
    ],
}

catalog_tactic = CatalogTactic(catalog_config)

result = catalog_tactic(
    "Which Electronics products are rated 4.5 or above? Show names and prices."
)
print_section("Catalog Result")
print(result)

In [ ]:
# The agent's typical workflow via the proxy:
#
# Turn 1: query_api_doc("catalog")
#   → learns about: catalog/products/list, catalog/products/detail, catalog/reviews/list
#
# Turn 2: run_python code:
#   products = CALL_API("catalog/products/list", {"category": "Electronics", "min_rating": 4.5})
#   print(products)
#
# Turn 3: format the output and return final answer

# Multi-step analysis example
result2 = catalog_tactic(
    "Find the highest-rated product, then fetch its reviews and summarize what customers say."
)
print_section("Review Summary")
print(result2)

---
## Section 5a — Interpreter Mode: The `run_python` / `CALL_API` Loop in Detail

With `exec_env: interpreter`, lllm automatically injects **two tools** into every agent that has a proxy config:

- **`query_api_doc(proxy_name)`** — returns the full endpoint listing for that proxy (lazy documentation)
- **`run_python(code)`** — executes Python in a **persistent in-process namespace**; `CALL_API(path, params)` is pre-injected

The agent's typical multi-turn session:

```
Turn 1:  query_api_doc("catalog")
         → endpoint listing injected into dialog as a tool result

Turn 2:  run_python("""
             products = CALL_API("catalog/products/list", {"category": "Electronics"})
             print(products)
         """)
         → stdout captured; products variable is now in the namespace

Turn 3:  run_python("""
             # products is still in scope from Turn 2
             best = max(products, key=lambda p: p["rating"])
             reviews = CALL_API("catalog/reviews/list", {"product_id": best["id"]})
             print(f"{best['name']}: {len(reviews)} reviews")
         """)
         → multi-step analysis building on previous state

Turn 4:  model writes the final answer based on all outputs seen so far
```

Key behaviors of the interpreter:

| Behavior | Detail |
|----------|--------|
| Namespace persistence | Variables survive across `run_python` calls within the same `agent.respond()` session |
| Output capture | Only `print()` output is returned — the last expression is **not** auto-returned |
| Exception handling | Uncaught exceptions are returned as formatted tracebacks so the agent can diagnose and retry |
| Output truncation | Output longer than `max_output_chars` is cut off with a truncation indicator; the agent is told |
| Timeout | Each `run_python` call runs in a daemon thread; `TimeoutError` if it exceeds `timeout` seconds |
| Repeated-call detection | If the model calls the same tool with the same args twice, lllm injects a warning to break the loop |

In [ ]:
# What query_api_doc returns for the CatalogProxy defined in Section 5.
# Simulate it here so you can see the format without a live tactic call.

from lllm.proxies import ProxyManager

# Build a manager that wraps our CatalogProxy
manager = ProxyManager(activate_proxies=["catalog"], deploy_mode=False)

doc = manager.query_api_doc("catalog")
print("=== query_api_doc('catalog') output ===")
print(doc)

In [ ]:
# Demonstrate the persistent namespace directly using AgentInterpreter.
# This is the same interpreter the proxy system uses internally.
from lllm.proxies.interpreter import AgentInterpreter

# Create an interpreter with CALL_API wired to our CatalogProxy manager
interp = AgentInterpreter(proxy_manager=manager, max_output_chars=3000, timeout=30.0)

print("=== Turn 1: fetch Electronics products ===")
out1 = interp.run("""
products = CALL_API("catalog/products/list", {"category": "Electronics", "min_rating": 0.0})
print(f"Fetched {len(products)} Electronics products")
for p in products:
    print(f"  {p['id']}: {p['name']} — ${p['price']}  rating={p['rating']}")
""")
print(out1)

print("=== Turn 2: variables from Turn 1 are still in scope ===")
out2 = interp.run("""
# 'products' is still available — no need to fetch again
best = max(products, key=lambda p: p["rating"])
print(f"Highest-rated: {best['name']} (rating={best['rating']})")
""")
print(out2)

print("=== Turn 3: chain another CALL_API using the variable from Turn 2 ===")
out3 = interp.run("""
reviews = CALL_API("catalog/reviews/list", {"product_id": best["id"]})
print(f"Found {len(reviews)} review(s) for {best['name']}:")
for r in reviews:
    print(f"  [{r['rating']}/5] {r['text'][:80]}")
""")
print(out3)

In [ ]:
# Exceptions are returned as tracebacks — the agent can read them and fix its code
print("=== Exception handling — bad CALL_API call ===")
out_err = interp.run("""
# This endpoint does not exist — will produce an error
result = CALL_API("catalog/products/nonexistent", {"foo": "bar"})
print(result)
""")
print(out_err)  # formatted traceback, not a crash

# Output truncation
print("\n=== Output truncation ===")
truncating_interp = AgentInterpreter(proxy_manager=manager, max_output_chars=100, timeout=30.0)
out_trunc = truncating_interp.run("""
for i in range(50):
    print(f"Line {i}: some output that will be truncated after max_output_chars")
""")
print(out_trunc)  # cut off at 100 chars with truncation indicator

In [ ]:
# Full proxy config reference — all available options
proxy_config_reference = """
proxy:
  activate_proxies: [catalog, fmp, exa]  # which proxies to load; empty = all registered
  deploy_mode: false                      # passed to proxy instances (e.g. for rate limiting)
  cutoff_date: "2026-01-01"              # ISO date; proxies can filter data by this
  exec_env: interpreter                   # "interpreter" | "jupyter" | null
  max_output_chars: 5000                  # truncate run_python stdout; 0 = no limit
  truncation_indicator: "... (truncated)"
  timeout: 60.0                           # seconds per run_python call; interpreter only
  prompt_template: null                   # custom template; null = auto-select

# Multi-proxy setup: analyst sees all three proxies
# The agent lazily fetches docs for whichever it needs via query_api_doc
agent_configs:
  - name: research_analyst
    system_prompt: >
      You are a financial research analyst with access to market data,
      macroeconomic indicators, and news search. Use them together.
    proxy:
      activate_proxies: [fmp, fred, exa]
      exec_env: interpreter
      max_output_chars: 8000
      timeout: 90.0
"""
print(proxy_config_reference)

# Note on global vs per-agent proxy config:
# A 'proxy:' block under 'global:' applies to every agent as a default.
# Per-agent 'proxy:' blocks deep-merge on top of global, field by field.
print("Per-agent proxy config deep-merges on top of global proxy config.")

---
## Section 5b — Jupyter Notebook Sandbox

The **Jupyter sandbox** is the alternative to interpreter mode for tasks that need notebook output artifacts, Matplotlib/Plotly visualizations, or a cell-by-cell execution history.

### How it differs from interpreter mode

| | Interpreter (`exec_env: interpreter`) | Jupyter (`exec_env: jupyter`) |
|--|--------------------------------------|-------------------------------|
| Execution | In-process `exec()` — zero overhead | Subprocess kernel (~2–3s startup) |
| Parallel agents | Safe — isolated namespaces per agent | Heavy — one subprocess per agent |
| Output artifacts | None | `.ipynb` notebook file |
| Visualizations | Not available | Full matplotlib / plotly support |
| Audit trail | None | Cell-by-cell execution history |
| Who drives execution | lllm (automatic tool loop) | **Your tactic** (explicit cell extraction) |
| Best for | Data retrieval, API orchestration, computation | Exploratory analysis, charts, reports |

### The cell-tag interface

In Jupyter mode, lllm **does not** inject `run_python`. Instead, the agent writes structured XML tags and your tactic extracts and executes them:

```
<python_cell>
import pandas as pd
data = CALL_API("catalog/products/list", {})
df = pd.DataFrame(data)
print(df.head())
</python_cell>

<markdown_cell>
## Product Summary
The table above shows all available products.
</markdown_cell>

<TERMINATE_NOTEBOOK>   ← agent sends this alone when the analysis is complete
```

Your tactic drives the loop: extract cells → run in `JupyterSession` → feed output back → repeat.

In [ ]:
# Step 1: System prompt and parser for the notebook interface.
# The system prompt explains the cell-tag protocol to the model.
# The parser enforces it — raising ParseError if the model forgets the tags.

from lllm import Prompt
from lllm.utils import find_all_xml_tags_sorted
from lllm.core.const import ParseError

NOTEBOOK_SYSTEM = """
You are a data analyst working in a Jupyter Notebook.

## Interface
- Write Python code inside `<python_cell>...</python_cell>` tags.
- Write narrative and summaries inside `<markdown_cell>...</markdown_cell>` tags.
- When your analysis is complete, respond with `<TERMINATE_NOTEBOOK>` alone on a line (no cells).

## Rules
- Cells execute sequentially; variables and imports persist across turns.
- If a cell fails you will see the traceback — fix **only that cell** and resubmit.
- Use `CALL_API(endpoint, params)` to query data. Always call `query_api_doc` first
  to discover available endpoints and their parameter formats.
- Always include at least one markdown cell summarizing your findings before terminating.
"""


def notebook_parser(message: str, **kwargs) -> dict:
    """Extract python_cell / markdown_cell tags and detect TERMINATE_NOTEBOOK."""
    matches = find_all_xml_tags_sorted(message)
    terminate = "<TERMINATE_NOTEBOOK>" in message

    cells = [
        (m["tag"], m["content"])
        for m in matches
        if m["tag"] in ("python_cell", "markdown_cell")
    ]

    # Must have cells OR be a termination message — never both empty
    if not cells and not terminate:
        raise ParseError(
            "No cells found and no termination signal. "
            "Wrap code in <python_cell>...</python_cell>, "
            "markdown in <markdown_cell>...</markdown_cell>, "
            "or respond with <TERMINATE_NOTEBOOK> when done."
        )

    # If TERMINATE is mixed with python cells, ignore the termination
    if terminate and any(tag == "python_cell" for tag, _ in cells):
        terminate = False

    return {"raw": message, "cells": cells, "terminate": terminate}


analyst_prompt = Prompt(
    path="tutorial/notebook_analyst",
    prompt=NOTEBOOK_SYSTEM,
    parser=notebook_parser,
)

print("Notebook system prompt and parser defined.")

In [ ]:
# Step 2: Agent config — exec_env: jupyter injects only query_api_doc.
# run_python is NOT injected; the tactic drives execution via JupyterSession.

from lllm import Agent, Tactic
from lllm.invokers import build_invoker

jupyter_agent_config = {
    "global": {
        "model_name": SMART_MODEL,          # use smarter model for multi-step reasoning
        "model_args": {"temperature": 0.1},
    },
    "agent_configs": [
        {
            "name": "analyst",
            "system_prompt": NOTEBOOK_SYSTEM,
            "proxy": {
                "activate_proxies": ["catalog"],
                "exec_env": "jupyter",          # key difference from Section 5
            },
        },
    ],
}

print("Agent config with exec_env: jupyter")
print("Only query_api_doc will be injected — run_python is NOT available.")
print("The tactic extracts <python_cell> tags and runs them in JupyterSession.")

In [ ]:
# Step 3: Write the tactic that drives the cell-execution loop.
# This is the core pattern for Jupyter mode — your tactic owns the execution.

import os, tempfile

class NotebookAnalysisTactic(Tactic):
    """
    Agent writes <python_cell> / <markdown_cell> tags.
    Tactic extracts them, runs them in a JupyterSession, feeds output back.
    Loop ends when agent sends <TERMINATE_NOTEBOOK>.
    Returns the path to the saved .ipynb file.
    """
    name = "notebook_analysis"
    agent_group = ["analyst"]
    MAX_STEPS = 12   # safety cap on the number of agent turns

    def call(self, task: str, output_dir: str = None, **kwargs) -> str:
        from lllm.sandbox.jupyter import JupyterSession

        agent = self.agents["analyst"]
        nb_dir = output_dir or tempfile.mkdtemp(prefix="lllm_nb_")

        # Initialise a JupyterSession.
        # init_session() wires CALL_API into the kernel namespace automatically
        # when the agent config has a proxy block.
        session = JupyterSession(
            name="analysis",
            dir=nb_dir,
            metadata={
                "proxy": {
                    "activate_proxies": ["catalog"],
                    "deploy_mode": False,
                }
            },
        )
        session.init_session()

        agent.open("main")
        agent.receive(task)

        for step in range(self.MAX_STEPS):
            msg = agent.respond()
            parsed = msg.parsed

            if parsed["terminate"]:
                print(f"Agent terminated after {step + 1} turn(s).")
                break

            # Execute each cell and collect outputs
            outputs = []
            for tag, code in parsed["cells"]:
                if tag == "python_cell":
                    output = session.run_cell(code)

                    # If the cell failed, give the agent a chance to fix it
                    if session.last_cell_failed:
                        agent.receive(
                            f"The cell failed with this error:\n\n{output}\n\n"
                            "Please fix only the failing cell. "
                            "Provide a single <python_cell> with the corrected code."
                        )
                        fix_msg = agent.respond()
                        if fix_msg.parsed["cells"]:
                            _, fixed_code = fix_msg.parsed["cells"][0]
                            output = session.run_cell(fixed_code)

                    outputs.append(f"[cell output]\n{output}")
                else:
                    session.add_markdown_cell(code)

            # Feed all cell outputs back so the agent can continue
            if outputs:
                agent.receive("\n\n".join(outputs))
        else:
            print(f"Warning: reached MAX_STEPS ({self.MAX_STEPS}) without termination.")

        nb_path = session.notebook_path
        print(f"Notebook saved to: {nb_path}")
        return nb_path


print("NotebookAnalysisTactic defined.")

In [ ]:
# Step 4: Run the tactic.
# The agent will:
#   1. Call query_api_doc("catalog") to learn the API
#   2. Write <python_cell> blocks that CALL_API and print results
#   3. Write <markdown_cell> blocks with narrative
#   4. Send <TERMINATE_NOTEBOOK> when done
#
# Requires: pip install jupyter nbformat ipykernel

nb_dir = tempfile.mkdtemp(prefix="lllm_nb_tutorial_")

notebook_tactic = NotebookAnalysisTactic(jupyter_agent_config)

notebook_path = notebook_tactic(
    task=(
        "Analyse the product catalog. "
        "Show which categories are available, compare average ratings and prices per category, "
        "and identify the top-3 highest-rated products. "
        "Write a brief conclusion in a markdown cell."
    ),
    output_dir=nb_dir,
)

print(f"\nDone! Open the notebook:")
print(f"  jupyter notebook {notebook_path}")

In [ ]:
# Step 5: Bring your own execution environment.
# The cell-tag pattern doesn't require JupyterSession.
# Any object with run_cell() / add_markdown_cell() / last_cell_failed works.

class InMemorySandbox:
    """
    Minimal sandbox: executes Python cells in a shared dict namespace.
    Acts as a drop-in replacement for JupyterSession in the tactic above.
    No subprocess, no kernel — just exec() with a persistent namespace.
    """
    def __init__(self, proxy_manager=None):
        self._ns = {}
        if proxy_manager:
            self._ns["CALL_API"] = proxy_manager.call_api
        self.last_cell_failed = False
        self._outputs: list[str] = []

    def init_session(self):
        pass  # nothing to start

    def run_cell(self, code: str) -> str:
        import io, contextlib
        buf = io.StringIO()
        self.last_cell_failed = False
        try:
            with contextlib.redirect_stdout(buf):
                exec(code, self._ns)  # noqa: S102
        except Exception as exc:
            self.last_cell_failed = True
            return f"Error: {type(exc).__name__}: {exc}"
        return buf.getvalue()

    def add_markdown_cell(self, text: str) -> None:
        self._outputs.append(f"[markdown] {text[:80]}")

    @property
    def notebook_path(self) -> str:
        return "(in-memory sandbox — no .ipynb file)"


# Swap JupyterSession for InMemorySandbox in the tactic to test without a kernel
class InMemoryNotebookTactic(NotebookAnalysisTactic):
    name = "in_memory_notebook"

    def call(self, task: str, **kwargs) -> str:
        agent = self.agents["analyst"]
        session = InMemorySandbox(proxy_manager=manager)  # manager from Section 5a
        session.init_session()

        agent.open("main")
        agent.receive(task)

        for step in range(self.MAX_STEPS):
            msg = agent.respond()
            parsed = msg.parsed
            if parsed["terminate"]:
                print(f"Terminated after {step + 1} turns.")
                break
            outputs = []
            for tag, code in parsed["cells"]:
                if tag == "python_cell":
                    output = session.run_cell(code)
                    if session.last_cell_failed:
                        agent.receive(
                            f"Cell failed:\n{output}\n\n"
                            "Fix it in a single <python_cell>."
                        )
                        fix = agent.respond()
                        if fix.parsed["cells"]:
                            output = session.run_cell(fix.parsed["cells"][0][1])
                    outputs.append(f"[output]\n{output}")
                else:
                    session.add_markdown_cell(code)
            if outputs:
                agent.receive("\n\n".join(outputs))

        return session.notebook_path


in_mem_tactic = InMemoryNotebookTactic(jupyter_agent_config)
result = in_mem_tactic("List all products and their categories, sorted by rating descending.")
print("Result:", result)

---
## Section 6 — `lllm.toml`, YAML Config, and Auto-Discovery

As projects grow, hard-coded prompts and inline configs become hard to manage.  
lllm's config system lets you declare everything in files and auto-discover them at startup.

### Typical project layout

```
my_project/
├── lllm.toml           ← project declaration (scanned paths, package metadata)
├── lllm_packages/      ← drop third-party packages here (auto-discovered)
├── prompts/
│   └── analyst.py      ← Prompt objects (auto-discovered)
├── configs/
│   └── default.yaml    ← agent configs (auto-discovered)
└── tactics/
    └── analyzer.py     ← Tactic subclasses (auto-discovered)
```

### `lllm.toml` minimal config

```toml
[package]
name = "my_project"
version = "0.1.0"

[prompts]
paths = ["prompts/"]

[configs]
paths = ["configs/"]

[tactics]
paths = ["tactics/"]
```

### What gets discovered automatically

| File | Registered as |
|------|---------------|
| `.py` with `Prompt` instances | prompts |
| `.py` with `BaseProxy` subclasses | proxies |
| `.py` with `Tactic` subclasses | tactics |
| `.yaml` / `.yml` in `configs/` | configs |

In [ ]:
# Using auto-discovery: load_package() reads lllm.toml and discovers everything
from lllm import load_package, resolve_config, build_tactic, load_prompt

# In a real project with a lllm.toml:
# load_package()  # discovers prompts, tactics, configs in one call

# After discovery, load resources by name:
# prompt = load_prompt("analyst/system")          # by registered path
# prompt = load_prompt("my_project:analyst/system") # fully qualified

# Load and resolve a YAML config (merges inheritance, resolves prompt paths)
# config = resolve_config("default")
# tactic = build_tactic(config, name="analyzer")

print("Auto-discovery is triggered by load_package() or an lllm.toml in the project root.")
print("The tutorials/packages/review_analyzer/ folder is a real example — see Section 15.")

In [ ]:
# YAML agent config format (what configs/default.yaml looks like)
yaml_config_example = """
tactic_type: analyzer

global:
  model_name: gpt-4o
  model_args:
    temperature: 0.1
  context_manager:
    type: default
    max_tokens: 128000

agent_configs:
  - name: extractor
    system_prompt_path: analyst/system   # resolves to registered Prompt
    model_args:
      max_completion_tokens: 4000

  - name: synthesizer
    system_prompt: "You are a concise writer."  # inline is also fine
    model_name: gpt-4o-mini              # override model for this agent only
"""
print("Example YAML config:")
print(yaml_config_example)

---
## Section 7 — Config Inheritance and `vendor_config`

YAML configs support inheritance via the `base` key — deep-merged recursively.  
`vendor_config` lets you import another package's config and apply local overrides.

In [ ]:
# Config inheritance example
base_config_yaml = """
# configs/base.yaml
global:
  model_name: gpt-4o
  model_args:
    temperature: 0.1
    seed: 42

agent_configs:
  - name: writer
    system_prompt: "You are a technical writer."
"""

fast_config_yaml = """
# configs/fast.yaml
base: base   # inherits from base.yaml

global:
  model_name: gpt-4o-mini  # override model; temperature/seed are kept from base
"""

print("base.yaml:")
print(base_config_yaml)
print("fast.yaml (inherits base):")
print(fast_config_yaml)

# resolve_config("fast") deep-merges fast.yaml on top of base.yaml
# The effective config has model_name=gpt-4o-mini, temperature=0.1, seed=42

In [ ]:
# vendor_config — import a dependency's config and override it
# Useful when your project depends on another lllm package
from lllm import vendor_config

# Example: use review_analyzer's default config but swap the model
# vendored = vendor_config("review_analyzer:default", overrides={
#     "global": {"model_name": "claude-sonnet-4-6"}
# })
# The vendored dict is standalone — no longer requires the dependency to be installed

print("vendor_config() materialises a dependency's config as a standalone dict.")
print("This makes your build self-contained — no transitive lllm package dependency at runtime.")

In [ ]:
# Virtual folder prefixes (under) — register resources under a different path
under_example = """
# lllm.toml excerpt
[prompts]
paths = [
    { path = "prompts/v2/", under = "v2" }
]

# A file at prompts/v2/greeter.py with path="greeter/system"
# will be registered as "v2/greeter/system"
# Load it with: load_prompt("v2/greeter/system")
"""
print(under_example)

---
## Section 8 — Skills: Progressive Capability Disclosure

> **Experimental** — follows the [agentskills.io](https://agentskills.io) open standard.

Skills are reusable capability packages attached to agents via config.  
Instead of loading all instructions upfront, lllm:
1. Injects only skill names + one-sentence descriptions (~50-100 tokens each)
2. Provides an `activate_skill` tool so the model pulls full instructions on demand

This keeps context lean for agents with many installed skills.

In [ ]:
# Skills in YAML config — three ways to declare them
skills_yaml = """
global:
  model_name: claude-sonnet-4-6
  skills: [pdf, commit]          # all agents get these by default

agent_configs:
  - name: coder
    system_prompt_path: system/coder
    skills: [commit, code-review] # replaces (not merges) the global list

  - name: writer
    system_prompt_path: system/writer
    # inherits global: skills: [pdf, commit]
"""

print("Skills can be declared as:")
print()
rows = [
    ("Local name",        "pdf",                          "Scanned from .agents/skills/ or ~/.agents/skills/"),
    ("Anthropic skill ID","skill_01abc123",               "Passed to Anthropic API; content injected server-side"),
    ("URL",               "https://example.com/SKILL.md", "Downloaded at agent build time"),
    ("Wildcard",          '"*"',                          "Load ALL locally discovered skills"),
]
for fmt, example, desc in rows:
    print(f"  {fmt:20} {example:40} {desc}")

In [ ]:
# What the model sees in its context (a compact listing)
context_snippet = """
<available_skills>
  <skill name="data-analysis">
    <description>
      Analyse tabular data, compute statistics, identify trends.
      Use when working with CSV or numerical datasets.
    </description>
  </skill>
  <skill name="code-review">
    <description>
      Review Python code for bugs, style, and security issues.
      Use when asked to review or audit code.
    </description>
  </skill>
</available_skills>
"""
print("What the model sees (injected into system prompt):")
print(context_snippet)
print("The model calls activate_skill('data-analysis') to get the full instructions on demand.")

---
## Section 9 — Creating a Local Skill

A skill is a directory under `.agents/skills/` (project-local) or `~/.agents/skills/` (user-global), containing a `SKILL.md` file with YAML frontmatter.

In [ ]:
import pathlib

# Create a sample skill in the tutorials folder
skill_dir = pathlib.Path("..") / ".agents" / "skills" / "product-analysis"
skill_dir.mkdir(parents=True, exist_ok=True)

skill_md = skill_dir / "SKILL.md"
skill_md.write_text(
    """---
name: product-analysis
description: >
  Analyse product catalogs, reviews, and pricing data.
  Use when working with e-commerce datasets or product research tasks.
---

# Product Analysis

Follow these steps when analysing product data:

1. **Identify the scope** — determine what products, categories, or time range is relevant.
2. **Fetch structured data** — use the catalog API to get ratings, prices, and reviews.
3. **Compute statistics** — calculate averages, ranges, and distributions.
4. **Identify trends** — compare across categories, time, or user segments.
5. **Summarize findings** — lead with the most actionable insight.

## Output format

Always return:
- A one-sentence executive summary
- Three bullet points of key findings
- A recommended action for the product team
"""
)

print(f"Skill created at: {skill_md.resolve()}")
print("\nSKILL.md preview:")
print(skill_md.read_text()[:400])

In [ ]:
# The key point: write the description as a trigger, not a title.
# The model sees only the description at first. It decides to activate
# the skill based on whether the task matches.

# Bad description:
bad_desc = "Product Analysis skill"

# Good description (trigger-style):
good_desc = (
    "Analyse product catalogs, reviews, and pricing data. "
    "Use when working with e-commerce datasets or product research tasks."
)

print("Bad  :", bad_desc)
print("Good :", good_desc)

---
## Section 10 — Named Runtimes for Parallel Experiments

The default runtime is a global singleton. For A/B experiments or multi-tenant environments, **named runtimes** give each experiment its own isolated resource registry.

In [ ]:
from lllm import load_runtime, get_runtime, get_default_runtime

# Load dedicated runtimes from separate config directories
# Each runtime has its own isolated resource registry

# load_runtime("./configs/exp1/lllm.toml", name="experiment_1")
# load_runtime("./configs/exp2/lllm.toml", name="experiment_2")

# rt1 = get_runtime("experiment_1")
# rt2 = get_runtime("experiment_2")

# Build tactics against specific runtimes to isolate them completely
# tactic_v1 = MyTactic(config_v1, runtime=rt1)
# tactic_v2 = MyTactic(config_v2, runtime=rt2)

# Access the default runtime
default_rt = get_default_runtime()
print("Default runtime type:", type(default_rt).__name__)

print("\nNamed runtimes are useful for:")
print("  - A/B testing two model versions without cross-contamination")
print("  - Multi-tenant systems where each tenant has different prompt sets")
print("  - Running integration tests against a clean registry")

---
## Section 11 — Streaming Responses

Attach a `stream_handler` to an `Agent` to receive tokens as they are generated.  
This is useful for interactive UIs where you want to show text progressively.

In [ ]:
from lllm.invokers.base import BaseStreamHandler

class PrintStreamHandler(BaseStreamHandler):
    """Print each token to stdout as it arrives."""

    def __init__(self):
        self.tokens: list[str] = []

    def on_token(self, token: str) -> None:
        print(token, end="", flush=True)
        self.tokens.append(token)

    def on_done(self) -> None:
        print()  # newline after all tokens


stream_handler = PrintStreamHandler()

stream_agent = Agent(
    name="streamer",
    system_prompt=Prompt(
        path="tutorial/storyteller",
        prompt="You are a storyteller. Write vivid, short stories.",
    ),
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    stream_handler=stream_handler,
)

print("Streaming response:")
print("-" * 60)
stream_agent.open("story")
stream_agent.receive("Write a two-sentence story about a robot who discovers music.")
msg = stream_agent.respond()  # tokens print to stdout as they arrive
print("-" * 60)
print(f"\nTotal tokens streamed: {len(stream_handler.tokens)}")

In [ ]:
# Advanced stream handler: collect tokens and broadcast to multiple sinks
class CollectingStreamHandler(BaseStreamHandler):
    """Collect tokens into a buffer; optionally broadcast to multiple handlers."""

    def __init__(self, *child_handlers: BaseStreamHandler):
        self.buffer = []
        self.child_handlers = child_handlers

    def on_token(self, token: str) -> None:
        self.buffer.append(token)
        for h in self.child_handlers:
            h.on_token(token)

    def on_done(self) -> None:
        for h in self.child_handlers:
            h.on_done()

    @property
    def text(self) -> str:
        return "".join(self.buffer)


print_handler = PrintStreamHandler()
collect_handler = CollectingStreamHandler(print_handler)

stream_agent2 = Agent(
    name="streamer2",
    system_prompt=Prompt(path="tutorial/poet", prompt="You are a poet."),
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    stream_handler=collect_handler,
)

print("Streaming poem:")
print("-" * 60)
stream_agent2.open("poem")
stream_agent2.receive("Write a haiku about artificial intelligence.")
stream_agent2.respond()
print("-" * 60)
print(f"Collected {len(collect_handler.buffer)} token chunks")

---
## Section 12 — Image (Multimodal) Input

Agents support multimodal conversations via `receive_image()`.  
Accepted formats: file path, PIL `Image` object, or base64-encoded string.

In [ ]:
# Create a sample PNG to demo with (requires Pillow: pip install Pillow)
try:
    from PIL import Image, ImageDraw
    import io

    # Create a simple chart-like image
    img = Image.new("RGB", (400, 200), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    # Draw a simple bar chart
    bars = [("Q1", 80), ("Q2", 120), ("Q3", 95), ("Q4", 150)]
    for i, (label, value) in enumerate(bars):
        x = 40 + i * 90
        draw.rectangle([x, 180 - value, x + 60, 180], fill=(70, 130, 180))
        draw.text((x + 15, 185), label, fill=(0, 0, 0))

    img_path = "/tmp/sample_chart.png"
    img.save(img_path)
    print(f"Sample image saved to: {img_path}")
    PILLOW_AVAILABLE = True
except ImportError:
    print("Pillow not installed (pip install Pillow). Skipping image creation.")
    PILLOW_AVAILABLE = False

---
## Summary

| Advanced Feature | Key API |
|-----------------|----------|
| Tools | `@tool(description=..., prop_desc={...})` |
| Tool result format | `@tool(processor=my_fn)` |
| Manual schema | `Function(name=..., properties={...})` |
| Link implementation | `fn.link_function(callable)` |
| MCP server | `Prompt(mcp_servers_list=[MCP(...)])` |
| Define a proxy | `@ProxyRegistrator(path=...) + @BaseProxy.endpoint(...)` |
| Wire proxy (interpreter) | `proxy: {activate_proxies: [...], exec_env: interpreter}` |
| Inspect API docs | `ProxyManager(...).query_api_doc("name")` |
| Run code with CALL_API | `AgentInterpreter(proxy_manager=...).run(code)` |
| Persistent namespace | Variables survive across `.run()` calls within a session |
| Wire proxy (Jupyter) | `proxy: {activate_proxies: [...], exec_env: jupyter}` |
| Notebook system prompt | `<python_cell>`, `<markdown_cell>`, `<TERMINATE_NOTEBOOK>` tags |
| Notebook parser | Custom `parser` function using `find_all_xml_tags_sorted` |
| Drive Jupyter loop | `JupyterSession.init_session()` → `run_cell()` → feed output back |
| Cell error recovery | Check `session.last_cell_failed` → re-prompt → `run_cell(fixed)` |
| Custom sandbox | Any object with `run_cell()`, `add_markdown_cell()`, `last_cell_failed` |
| Auto-discovery | `load_package()` reads `lllm.toml` |
| Load discovered resource | `load_prompt("path")`, `resolve_config("name")` |
| Config inheritance | `base: parent_name` in YAML |
| Import external config | `vendor_config("pkg:config", overrides={...})` |
| Skills (progressive) | `skills: [pdf, commit]` in YAML config |
| Create local skill | `.agents/skills/<name>/SKILL.md` |
| Named runtimes | `load_runtime(path, name=...)` |
| Streaming | `Agent(stream_handler=MyHandler())` |
| Image input | `agent.receive_image(path_or_b64, caption=...)` |
| Custom retry | Subclass `DefaultSimpleHandler`, override `on_exception` |
| Batch with errors | `tactic.bcall(tasks, fail_fast=False)` |

### What to explore next

- **Computer Use Agent** — `lllm.tools.cua` for browser automation via Playwright
- **Responses API** — `api_type = "response"` per agent for native OpenAI web search
- **Full package example** — `examples/code_review_service/` for a production FastAPI wrapper
- **Analysis dashboard** — roadmap item: Streamlit/Dash GUI for the `LogStore` database

In [ ]:
# receive_image also accepts base64 strings
if PILLOW_AVAILABLE:
    import base64

    with open(img_path, "rb") as f:
        b64_image = base64.b64encode(f.read()).decode()

    vision_agent2 = Agent(
        name="vision2",
        system_prompt=Prompt(path="tutorial/vision2", prompt="You are a visual analyst."),
        model=SMART_MODEL,
        llm_invoker=build_invoker({"invoker": "litellm"}),
    )

    vision_agent2.open("b64_test")
    vision_agent2.receive_image(b64_image, caption="Revenue chart (base64)")
    vision_agent2.receive("Describe what you see in one sentence.")
    r2 = vision_agent2.respond()
    print("Base64 image result:", r2.content)

---
## Section 13 — Custom Exception and Interrupt Handlers

When a `ParseError` is raised (from a parser or from your code), lllm retries automatically by sending an error prompt back to the model. You can **customize the retry prompt** by subclassing `DefaultSimpleHandler`.

This lets you adapt the retry strategy — providing more context, changing the tone, or escalating after repeated failures.

In [ ]:
from lllm.core.prompt import DefaultSimpleHandler, DefaultTagParser
from lllm.core.const import ParseError


class VerboseRetryHandler(DefaultSimpleHandler):
    """Escalates the retry message with each successive failure."""

    def on_exception(self, prompt: Prompt, session) -> Prompt:
        retry_num = session.exception_retries_count
        urgency = ["Please try again.", "This is very important.", "CRITICAL: fix this now."][min(retry_num, 2)]

        return prompt.extend(
            path=f"__verbose_retry_{retry_num}",
            prompt=(
                f"Attempt {retry_num + 1}: Your previous response had a formatting error: "
                "{{error_message}}. "
                f"{urgency} "
                "Wrap the decision in <decision>approve</decision> or <decision>reject</decision>."
            ),
        )


# A prompt that requires a <decision> tag
decision_prompt = Prompt(
    path="tutorial/decision_maker",
    prompt=(
        "You are a content moderator. Review the text and decide: approve or reject. "
        "Always include your decision in a <decision>approve</decision> or "
        "<decision>reject</decision> tag."
    ),
    parser=DefaultTagParser(
        xml_tags=["decision"],
        required_xml_tags=["decision"],
    ),
    handler=VerboseRetryHandler(),
)

decision_agent = Agent(
    name="moderator",
    system_prompt=decision_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    max_exception_retry=3,
)

samples = [
    "This is a great tutorial about machine learning!",
    "This content contains inappropriate language and hate speech.",
    "A neutral article about the weather forecast for tomorrow.",
]

for sample in samples:
    decision_agent.open("moderate")
    decision_agent.receive(sample)
    m = decision_agent.respond()
    decision = m.parsed["xml_tags"].get("decision", ["unknown"])[0]
    print(f"[{decision.upper():8}] {sample[:70]}")
    decision_agent.close("moderate")

---
## Section 14 — High-Throughput Batch Processing

For production pipelines that need to process thousands of items, `bcall` with a tuned `max_workers` is the primary tool.  
Pair it with a `LogStore` for full observability.

In [ ]:
import time
from lllm.logging import sqlite_store


class ReviewClassifierTactic(Tactic):
    """Classify a product review as positive, neutral, or negative."""
    name = "review_classifier"
    agent_group = ["classifier"]

    def call(self, task: str, **kwargs) -> str:
        agent = self.agents["classifier"]
        agent.open("classify")
        agent.receive(task)
        return agent.respond().content.strip()


batch_db = os.path.join(tempfile.gettempdir(), "lllm_batch.db")
batch_store = sqlite_store(batch_db)

batch_config = {
    "global": {"model_name": DEFAULT_MODEL, "model_args": {"temperature": 0.0}},
    "agent_configs": [
        {
            "name": "classifier",
            "system_prompt": (
                "You classify product reviews. Respond with exactly one word: "
                "POSITIVE, NEUTRAL, or NEGATIVE."
            ),
        },
    ],
}

batch_tactic = ReviewClassifierTactic(batch_config, log_store=batch_store)

# Build a large batch from our sample reviews (repeat to fill out the demo)
review_texts = [r["text"] for r in SAMPLE_REVIEWS] * 3   # 15 items
print(f"Processing {len(review_texts)} reviews...")

start = time.perf_counter()

# bcall with fail_fast=False — errors become exceptions in the result list
results = batch_tactic.bcall(
    review_texts,
    max_workers=5,
    fail_fast=False,
    tags={"batch_id": "demo-001", "env": "tutorial"},
)

elapsed = time.perf_counter() - start
print(f"Completed in {elapsed:.2f}s\n")

# Tally results
from collections import Counter
counts = Counter()
errors = []
for i, r in enumerate(results):
    if isinstance(r, Exception):
        errors.append((i, str(r)))
    else:
        counts[r] += 1

print("Classification counts:", dict(counts))
if errors:
    print(f"Errors: {len(errors)}")

# Aggregate cost across the whole batch
cost = batch_store.export_cost_summary(tags={"batch_id": "demo-001"})
print(f"\nBatch cost summary:")
print(json.dumps(cost, indent=2, default=str))

---
## Section 15 — Using a Pre-Built Package

The `tutorials/packages/review_analyzer/` folder is a self-contained lllm package:

```
review_analyzer/
├── lllm.toml           ← package declaration
├── prompts/
│   ├── analyzer.txt    ← analyzer system prompt
│   └── summarizer.txt  ← summarizer system prompt
└── tactic.py           ← ReviewAnalyzerTactic
```

This shows how to structure a real, shareable lllm package.

In [ ]:
from packages.review_analyzer.tactic import ReviewAnalyzerTactic

# The package has its own config — wire it to your preferred model
pkg_config = {
    "global": {"model_name": DEFAULT_MODEL},
    "agent_configs": [
        {"name": "analyzer", "system_prompt": (
            "You are a product review analyst. Identify sentiment (positive/neutral/negative), "
            "list up to 3 pros and 3 cons, and flag any defects or safety concerns. "
            "Be concise and factual."
        )},
        {"name": "summarizer", "system_prompt": (
            "You are a report writer. Given a review analysis, produce a 3-5 sentence "
            "executive summary with actionable insights for a product manager."
        )},
    ],
}

pkg_tactic = ReviewAnalyzerTactic(pkg_config)

# Run on real sample reviews
selected_reviews = [r["text"] for r in SAMPLE_REVIEWS[:3]]
output = pkg_tactic.run(reviews=selected_reviews)

print_section("Analysis")
print(output["analysis"][:500])

print_section("Executive Summary")
print(output["summary"])

In [ ]:
# Sharing a package with teammates
print("To export and share a package:")
print()
print("  lllm pkg export review_analyzer review_analyzer-v1.0.zip")
print()
print("To install it in another project:")
print()
print("  lllm pkg install review_analyzer-v1.0.zip")
print()
print("After installation, it lives in lllm_packages/ and is auto-discovered.")
print("Resources are available under the package namespace: review_analyzer:*")

---
## Section 16 — Architecture Summary

Here is the complete lllm architecture in ASCII:

```
lllm.toml                      ← project declaration
├── prompts/                   ← Prompt objects (auto-discovered)
├── configs/                   ← YAML agent configs (auto-discovered)
├── tactics/                   ← Tactic subclasses (auto-discovered)
├── proxies/                   ← BaseProxy subclasses (auto-discovered)
└── lllm_packages/             ← installed third-party packages

Runtime                        ← registry of all discovered resources
└── default / named            ← isolated registries for experiments

Tactic                         ← the program (orchestration logic)
└── call(task) → result
    ├── Agent A                ← LLM identity (name + model + system prompt)
    │   ├── Dialog "main"      ← append-only conversation history
    │   │   └── Message[]      ← role, content, cost, parsed
    │   └── Dialog "fork_1"    ← independent branch
    └── Agent B
        └── Dialog "session"

Prompt                         ← template + parser + tools + handlers
├── template string            ← {variable} placeholders
├── DefaultTagParser           ← XML tags, markdown blocks, signal tags
├── format=PydanticModel       ← structured JSON output
├── function_list=[...]        ← @tool or Function definitions
├── mcp_servers_list=[...]     ← MCP server connections
└── handler                    ← custom retry/interrupt behavior

LogStore                       ← persistence layer
└── TacticCallSession          ← per-call trace
    ├── AgentCallSession[]     ← per-agent call trace (interrupts, retries, cost)
    └── sub_tactic_sessions    ← nested tactic traces (cost rolls up)
```

In [ ]:
# The full production bootstrap pattern
from lllm import load_package, resolve_config, build_tactic
from lllm.logging import sqlite_store

print("Production bootstrap:")
print()
bootstrap_code = """
# 1. Load the project (reads lllm.toml, discovers everything)
load_package()

# 2. Create a log store
store = sqlite_store("./production.db")

# 3. Build a tactic from discovered config
config = resolve_config("default")   # loads configs/default.yaml with inheritance
tactic = build_tactic(config, log_store=store)

# 4. Run a tagged batch
results = tactic.bcall(
    my_tasks,
    max_workers=20,
    fail_fast=False,
    tags={"version": "1.2", "env": "prod"},
)
"""
print(bootstrap_code)

---
## Summary

| Advanced Feature | Key API |
|-----------------|----------|
| Tools | `@tool(description=..., prop_desc={...})` |
| Tool result format | `@tool(processor=my_fn)` |
| Manual schema | `Function(name=..., properties={...})` |
| Link implementation | `fn.link_function(callable)` |
| MCP server | `Prompt(mcp_servers_list=[MCP(...)])` |
| Define a proxy | `@ProxyRegistrator + @BaseProxy.endpoint` |
| Wire proxy to agent | `proxy: {activate_proxies: [...], exec_env: interpreter}` |
| Auto-discovery | `load_package()` reads `lllm.toml` |
| Load discovered resource | `load_prompt("path")`, `resolve_config("name")` |
| Config inheritance | `base: parent_name` in YAML |
| Import external config | `vendor_config("pkg:config", overrides={...})` |
| Skills (progressive) | `skills: [pdf, commit]` in YAML config |
| Create local skill | `.agents/skills/<name>/SKILL.md` |
| Named runtimes | `load_runtime(path, name=...)` |
| Streaming | `Agent(stream_handler=MyHandler())` |
| Image input | `agent.receive_image(path_or_b64, caption=...)` |
| Custom retry | Subclass `DefaultSimpleHandler`, override `on_exception` |
| Batch with errors | `tactic.bcall(tasks, fail_fast=False)` |

### What to explore next

- **Jupyter sandbox** — `exec_env: jupyter` + `JupyterSession` for notebook-generating agents
- **Computer Use Agent** — `lllm.tools.cua` for browser automation via Playwright
- **Responses API** — `api_type = "response"` per agent for native OpenAI web search
- **Full package example** — `examples/code_review_service/` for a production FastAPI wrapper